In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
print('test')

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ml")
 
 # Migration mapping: (old_gold_name, new_ml_name)
migrations = [
     ("gold.predictive_labels",                "ml.labels"),
     ("gold.phase2_selected_tags",             "ml.selected_tags"),
     ("gold.training_dataset_v2",              "ml.training_shortterm"),
     ("gold.training_dataset_daily_survival",  "ml.training_longterm"),
     ("gold.model_predictions",                "ml.predictions_shortterm"),
     ("gold.model_predictions_longterm_cox",   "ml.predictions_longterm"),
     ("gold.model_predictions_watchlist",      "ml.watchlist"),
     ("gold.shap_explanations",                "ml.drivers_shortterm"),
     ("gold.prediction_drivers_cox",           "ml.drivers_longterm"),
     ("gold.sensor_degradation_details",       "ml.degradation_longterm"),
     ("gold.aakr_training_memory",             "ml.aakr_memory"),
     ("gold.aakr_model_metadata",              "ml.aakr_metadata"),
     ("gold.aakr_scored",                      "ml.aakr_scores"),
     ("gold.aakr_episodes",                    "ml.aakr_episodes"),
     ("gold.aakr_health",                      "ml.aakr_health"),
     ("gold.anomaly_advisories",               "ml.anomaly_advisories"),
     ("gold.sensor_baselines",                 "ml.anomaly_baselines"),
     ("gold.multivariate_anomaly_scores",      "ml.anomaly_multivariate"),
 ]
 
for old, new in migrations:
     try:
         df = spark.table(old)
         cnt = df.count()
         df.write.mode("overwrite").format("delta").saveAsTable(new)
         print(f"â {old} â {new} ({cnt:,} rows)")
     except Exception as e:
         print(f"â {old} â {new}: {e}")
 
print("\n--- Migration complete. Verify with: SHOW TABLES IN ml ---")

In [ ]:
%%sql
drop table gold.dim_date

In [ ]:

df = spark.sql('select * from dbo.dim_date')
df.write.mode("overwrite").format("delta").saveAsTable('gold.dim_date')

In [ ]:
%%sql
select * from gold.dim_date

In [ ]:
from pyspark.sql import functions as F

fp = spark.table("gold.fact_pi")
rv2 = fp.filter(F.col("plant") == "RV2")

# 1. Search for pressure/load/drum/MW tags that could indicate running state
print("=== RV2 tags matching pressure/drum/load/MW patterns ===")
candidates = rv2.filter(
    F.col("Tag").rlike("(?i)(DRUM|PRESS|LOAD|MW$|MWNET|GROSS|STEAM|THROTTLE)")
).groupBy("Tag").agg(
    F.count("*").alias("rows"),
    F.avg("ValueNumeric").alias("avg"),
    F.min("ValueNumeric").alias("min_val"),
    F.max("ValueNumeric").alias("max_val"),
    F.stddev("ValueNumeric").alias("stddev"),
    F.percentile_approx("ValueNumeric", 0.5).alias("p50")
).orderBy(F.desc("rows"))
candidates.show(50, truncate=False)

# 2. All RV2 boiler tags summary (top 50 by volume)
print("=== All RV2_U2_Boiler tags (top 50 by row count) ===")
(rv2.filter(F.col("asset_id") == "RV2_U2_Boiler")
   .groupBy("Tag")
   .agg(F.count("*").alias("rows"),
        F.avg("ValueNumeric").alias("avg"),
        F.min("ValueNumeric").alias("min_val"),
        F.max("ValueNumeric").alias("max_val"))
   .orderBy(F.desc("rows"))
   .show(50, truncate=False))

# 3. Check if any RV2 tag has bimodal on/off behavior (good running indicator)
print("=== Tags with likely on/off bimodal distribution (high stddev relative to mean) ===")
bimodal = rv2.groupBy("Tag").agg(
    F.count("*").alias("rows"),
    F.avg("ValueNumeric").alias("avg"),
    F.stddev("ValueNumeric").alias("stddev"),
    F.min("ValueNumeric").alias("min_val"),
    F.max("ValueNumeric").alias("max_val"),
    F.percentile_approx("ValueNumeric", 0.05).alias("p5"),
    F.percentile_approx("ValueNumeric", 0.95).alias("p95")
).filter(
    (F.col("rows") > 10000) &
    (F.col("stddev") > 0.3 * F.abs(F.col("avg"))) &
    (F.col("max_val") > 5 * F.col("min_val") + 1)
).orderBy(F.desc("rows"))
bimodal.show(30, truncate=False)

In [ ]:
from pyspark.sql import functions as F
fp = spark.table("gold.fact_pi")
for tag in ["RV2:BTPU2BP21.AG", "RV2:BTPU2BP20.AG"]:
    sub = fp.filter(F.col("Tag") == tag)
    stats = sub.agg(
        F.count("*").alias("rows"),
        F.min("Timestamp").alias("min_ts"),
        F.max("Timestamp").alias("max_ts"),
        F.min("ValueNumeric").alias("min_val"),
        F.avg("ValueNumeric").alias("avg"),
        F.max("ValueNumeric").alias("max_val"),
        F.percentile_approx("ValueNumeric", 0.05).alias("p5"),
        F.percentile_approx("ValueNumeric", 0.50).alias("p50"),
        F.percentile_approx("ValueNumeric", 0.95).alias("p95")
    ).collect()[0]
    print(f"{tag}: {stats.rows:,} rows [{stats.min_ts} -> {stats.max_ts}]")
    if stats.rows > 0:
        print(f"  min={stats.min_val:.1f} p5={stats.p5:.1f} p50={stats.p50:.1f} p95={stats.p95:.1f} max={stats.max_val:.1f}")
        # Value distribution bands
        sub.withColumn("band",
            F.when(F.col("ValueNumeric") < 100, "< 100")
             .when(F.col("ValueNumeric") < 400, "100-400")
             .when(F.col("ValueNumeric") < 600, "400-600")
             .when(F.col("ValueNumeric") < 800, "600-800")
             .otherwise(">= 800")
        ).groupBy("band").agg(
            F.count("*").alias("rows"),
            F.round(F.count("*") * 100.0 / sub.count(), 1).alias("pct")
        ).orderBy("band").show(truncate=False)

In [ ]:
# Update gold.running_indicator: fix RV2 tag + fix RV3 turbine typo
from pyspark.sql import functions as F

ri = spark.table("gold.running_indicator")

# Fix RV2: BTPU2BPDRUM.AG -> BTPU2BP20.AG (Drum Press West, >600 PSI)
ri = ri.withColumn("Tag",
    F.when(F.col("Tag") == "RV2:BTPU2BPDRUM.AG", F.lit("RV2:BTPU2BP20.AG"))
     .otherwise(F.col("Tag"))
).withColumn("tag_description",
    F.when(F.col("Tag") == "RV2:BTPU2BP20.AG", F.lit("U2 Drum Pressure (West)"))
     .otherwise(F.col("tag_description"))
).withColumn("notes",
    F.when(F.col("Tag") == "RV2:BTPU2BP20.AG",
           F.lit("Original SME tag BTPU2BPDRUM.AG not in export. Using BTPU2BP20.AG (West drum). Threshold 600 PSI per SME spec."))
     .otherwise(F.col("notes"))
)

# Fix RV3 turbine: TXSU3TS15A.AG -> TXSU3TS15A1.AG (tag name typo)
ri = ri.withColumn("Tag",
    F.when(F.col("Tag") == "RV3:TXSU3TS15A.AG", F.lit("RV3:TXSU3TS15A1.AG"))
     .otherwise(F.col("Tag"))
).withColumn("notes",
    F.when(F.col("Tag") == "RV3:TXSU3TS15A1.AG",
           F.lit("Original tag TXSU3TS15A.AG not in export. Using TXSU3TS15A1.AG (confirmed 111k rows). Threshold 6 RPM per SME."))
     .otherwise(F.col("notes"))
)

ri.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold.running_indicator")
spark.table("gold.running_indicator").show(truncate=False)

In [ ]:
%%sql
select * from pi_tags_metadata where plant = 'RV2' --and Descriptor like '%pressure%' 

In [ ]:
# Investigate RV3 Boiler Feed Pump tag mappings

# Check PI metadata for feedwater flow tags
print("=== PI TAGS METADATA FOR FEEDWATER FLOW TAGS ===")
tags_df = spark.sql("""
    SELECT Name, Descriptor, EngineeringUnits
    FROM dbo.pi_tags_metadata
    WHERE Name LIKE '%FWFU3WF2%'
    ORDER BY Name
""")
tags_df.show(50, False)

print("\n=== BRIDGE TABLE MAPPINGS FOR RV3 BOILER FEED PUMP ===")
bridge_df = spark.sql("""
    SELECT Tag, asset_id, tag_role, tag_description
    FROM gold.bridge_pi_tag_to_asset
    WHERE asset_id LIKE '%Boiler_Feed_Pump%'
    ORDER BY asset_id, Tag
""")
bridge_df.show(100, False)

print("\n=== CHECK IF WEST PUMP ASSET EXISTS ===")
assets_df = spark.sql("""
    SELECT DISTINCT asset_id
    FROM gold.bridge_pi_tag_to_asset
    WHERE asset_id LIKE '%RV3%' AND asset_id LIKE '%Pump%'
    ORDER BY asset_id
""")
assets_df.show(50, False)

print("\n=== FEATURES USED IN PREDICTION FOR EAST PUMP ===")
features_df = spark.sql("""
    SELECT DISTINCT feature, tag_name, descriptor, engineering_units
    FROM gold.prediction_drivers_cox
    WHERE asset_id = 'RV3_U3_Boiler_Feed_Pump_East'
    ORDER BY feature
""")
features_df.show(100, False)

In [ ]:
# Check available tables
spark.sql("SHOW TABLES IN gold").filter("tableName LIKE '%gads%' OR tableName LIKE '%event%' OR tableName LIKE '%stop%'").show(50, False)

In [ ]:
# Check GADS fact table schema
spark.table("gold.fact_gads_event").printSchema()
spark.table("gold.fact_gads_event").show(5, False)